# Football Player Network Analysis using Apache Spark

Professional notebook structure.

In [61]:
from pyspark.sql import SparkSession

sparkSession = SparkSession.builder.appName("FootballPlaystyleAnalysis").master("local[*]").getOrCreate()
spark = sparkSession
sparkSession

Veri Setlerini Okuma

In [62]:
events = spark.read.option("multiline", "true").json("statsbomb-open-data/data/events/*.json")

26/07/24 11:11:57 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: statsbomb-open-data/data/events/*.json.
java.io.FileNotFoundException: File statsbomb-open-data/data/events/*.json does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apac

In [63]:
##Events kontrolleri
events.printSchema()

events.count()

len(events.columns)

events.show(5, truncate=False)

root
 |-- 50_50: struct (nullable = true)
 |    |-- outcome: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- bad_behaviour: struct (nullable = true)
 |    |-- card: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- ball_receipt: struct (nullable = true)
 |    |-- outcome: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- ball_recovery: struct (nullable = true)
 |    |-- offensive: boolean (nullable = true)
 |    |-- recovery_failure: boolean (nullable = true)
 |-- block: struct (nullable = true)
 |    |-- deflection: boolean (nullable = true)
 |    |-- offensive: boolean (nullable = true)
 |    |-- save_block: boolean (nullable = true)
 |-- carry: struct (nullable = true)
 |    |-- end_location: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |-- clearan

+-----+-------------+------------+-------------+-----+-----+---------+------------+-------+----+--------+--------------+--------+----------+--------+----------+------------------------------------+-----+---------------+------------+------------+------+----------+----------+----+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+------------------+-------------------------+----------+-------------------------------+----------+---------------+--------------------------------------+------+----+------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [64]:
##event types
from pyspark.sql.functions import col

events.select(col("type.name").alias("event_type")) \
      .distinct() \
      .orderBy("event_type") \
      .show(100, truncate=False)

+-----------------+
|event_type       |
+-----------------+
|50/50            |
|Bad Behaviour    |
|Ball Receipt*    |
|Ball Recovery    |
|Block            |
|Camera On        |
|Camera off       |
|Carry            |
|Clearance        |
|Dispossessed     |
|Dribble          |
|Dribbled Past    |
|Duel             |
|Error            |
|Foul Committed   |
|Foul Won         |
|Goal Keeper      |
|Half End         |
|Half Start       |
|Injury Stoppage  |
|Interception     |
|Miscontrol       |
|Offside          |
|Own Goal Against |
|Own Goal For     |
|Pass             |
|Player Off       |
|Player On        |
|Pressure         |
|Referee Ball-Drop|
|Shield           |
|Shot             |
|Starting XI      |
|Substitution     |
|Tactical Shift   |
+-----------------+



In [65]:
events.groupBy(
    col("type.name").alias("event_type")
).count().orderBy(col("count").desc()).show(100, truncate=False)

+-----------------+-------+
|event_type       |count  |
+-----------------+-------+
|Pass             |4103347|
|Ball Receipt*    |3844869|
|Carry            |3204878|
|Pressure         |1394727|
|Ball Recovery    |461754 |
|Duel             |310469 |
|Clearance        |196297 |
|Block            |168752 |
|Dribble          |146663 |
|Goal Keeper      |132397 |
|Miscontrol       |126295 |
|Foul Committed   |117581 |
|Foul Won         |111772 |
|Dispossessed     |110366 |
|Shot             |108282 |
|Interception     |96130  |
|Dribbled Past    |90736  |
|Substitution     |28044  |
|Injury Stoppage  |18172  |
|Half End         |17216  |
|Half Start       |17216  |
|50/50            |15838  |
|Tactical Shift   |11739  |
|Starting XI      |8470   |
|Referee Ball-Drop|6264   |
|Shield           |6007   |
|Player Off       |4541   |
|Player On        |4500   |
|Bad Behaviour    |2987   |
|Camera On        |2595   |
|Error            |2256   |
|Offside          |1513   |
|Camera off       |6

In [66]:
passes = events.filter(col("type.name") == "Pass")
passes.count()

4103347

In [67]:
## Extract the necessary columns for analysis
passes = passes.select(
    col("player.id").alias("passer_id"),
    col("player.name").alias("passer"),
    col("pass.recipient.id").alias("receiver_id"),
    col("pass.recipient.name").alias("receiver"),
    col("team.id").alias("team_id"),
    col("team.name").alias("team"),
    "minute",
    "second",
    "location",
    col("pass.end_location").alias("end_location")
)

##Show
passes.show(10, truncate=False)

+---------+-------------------------+-----------+-------------------------+-------+------+------+------+------------+------------+
|passer_id|passer                   |receiver_id|receiver                 |team_id|team  |minute|second|location    |end_location|
+---------+-------------------------+-----------+-------------------------+-------+------+------+------+------------+------------+
|5487     |Antoine Griezmann        |10481      |Aurélien Djani Tchouaméni|771    |France|0     |0     |[60.0, 40.0]|[48.4, 38.1]|
|10481    |Aurélien Djani Tchouaméni|24778      |Eduardo Camavinga        |771    |France|0     |2     |[47.9, 37.4]|[49.3, 28.7]|
|24778    |Eduardo Camavinga        |8519       |Dayotchanculle Upamecano |771    |France|0     |4     |[49.0, 25.2]|[38.1, 46.8]|
|8519     |Dayotchanculle Upamecano |3961       |N'Golo Kanté             |771    |France|0     |7     |[41.6, 49.4]|[49.5, 52.3]|
|3961     |N'Golo Kanté             |17592      |William Saliba           |771    |

In [68]:
passes.filter(col("receiver").isNull()).count()

256163

In [69]:
##Filtreyi uygula - alıcısı olmayan pasları çıkar
passes = passes.filter(col("receiver").isNotNull())
## Sayı kontrol
passes.count()

3847184

In [70]:
## create an edge list for the passes
edges = (
    passes.groupBy(
        "passer_id",
        "passer",
        "receiver_id",
        "receiver",
        "team_id",
        "team"
    )
    .count()
    .withColumnRenamed("count", "pass_count")

    
    
)

##kontrol
edges.show(20, truncate=False)

edges.count()

+---------+-----------------------------+-----------+------------------------------+-------+-------------------+----------+
|passer_id|passer                       |receiver_id|receiver                      |team_id|team               |pass_count|
+---------+-----------------------------+-----------+------------------------------+-------+-------------------+----------+
|8519     |Dayotchanculle Upamecano     |4445       |Jules Koundé                  |771    |France             |77        |
|5204     |Bruno Miguel Borges Fernandes|41092      |Nuno Mendes                   |780    |Portugal           |39        |
|3961     |N'Golo Kanté                 |3009       |Kylian Mbappé Lottin          |771    |France             |21        |
|10595    |Raphael Dias Belloli         |3063       |Danilo Luiz da Silva          |781    |Brazil             |20        |
|30486    |Pedro González López         |5203       |Sergio Busquets i Burgos      |772    |Spain              |60        |
|16022  

208518

In [71]:
## creating verteices

vertices = (
    passes.select(
        col("passer_id").alias("id"),
        col("passer").alias("name"),
        "team_id",
        "team"
    )
    .distinct()
)

vertices.count()
vertices.show(20, truncate=False)

+-----+--------------------------------+-------+------------------------+
|id   |name                            |team_id|team                    |
+-----+--------------------------------+-------+------------------------+
|3009 |Kylian Mbappé Lottin            |131    |Paris Saint-Germain     |
|6840 |Marcos Llorente Moreno          |772    |Spain                   |
|23725|Roman Bezus                     |911    |Ukraine                 |
|5487 |Antoine Griezmann               |212    |Atlético Madrid         |
|34639|Vitor Machado Ferreira          |131    |Paris Saint-Germain     |
|48396|Rocco Reitz                     |185    |Borussia Mönchengladbach|
|13620|Éder Gabriel Militão            |781    |Brazil                  |
|25305|Pedro Guilherme Abreu dos Santos|781    |Brazil                  |
|18618|Serhiy Kryvtsov                 |911    |Ukraine                 |
|31900|Oleksandr Karavaev              |911    |Ukraine                 |
|11396|Florian Grillitsch             

Matches'i okuma işlemi

In [72]:
matches = spark.read.option(
    "multiline", "true"
).json("statsbomb-open-data/data/matches/*/*.json")

26/07/24 11:13:47 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: statsbomb-open-data/data/matches/*/*.json.
java.io.FileNotFoundException: File statsbomb-open-data/data/matches/*/*.json does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at or

In [73]:
## Match kontrolleri

matches.select(
    "match_id",
    "competition.competition_name",
    "season.season_name",
    "home_team.home_team_name",
    "away_team.away_team_name",
    "match_date"
).show(10, truncate=False)

+--------+----------------+-----------+----------------------+--------------+----------+
|match_id|competition_name|season_name|home_team_name        |away_team_name|match_date|
+--------+----------------+-----------+----------------------+--------------+----------+
|3825848 |La Liga         |2015/2016  |Levante UD            |Eibar         |2015-09-23|
|3825895 |La Liga         |2015/2016  |Las Palmas            |Sevilla       |2015-09-23|
|3825894 |La Liga         |2015/2016  |RC Deportivo La Coruña|Getafe        |2016-05-01|
|3825855 |La Liga         |2015/2016  |Málaga                |Levante UD    |2016-05-02|
|3825908 |La Liga         |2015/2016  |Espanyol              |Eibar         |2016-05-15|
|3825883 |La Liga         |2015/2016  |Málaga                |Las Palmas    |2016-05-15|
|3825900 |La Liga         |2015/2016  |Sporting Gijón        |Villarreal    |2016-05-15|
|3825902 |La Liga         |2015/2016  |Rayo Vallecano        |Levante UD    |2016-05-15|
|3825876 |La Liga    

In [74]:
from pyspark.sql.functions import input_file_name

events = spark.read.option(
    "multiline", "true"
).json("statsbomb-open-data/data/events/*.json") \
.withColumn("file", input_file_name())

26/07/24 11:13:48 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: statsbomb-open-data/data/events/*.json.
java.io.FileNotFoundException: File statsbomb-open-data/data/events/*.json does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apac

In [75]:
##vertex ve edge sayısı
vertices.count(), edges.count()

(13256, 208518)

In [76]:
## en çok pas alan ve atan oyuncular
from pyspark.sql.functions import sum

edges.groupBy("passer") \
     .agg(sum("pass_count").alias("total_passes")) \
     .orderBy(col("total_passes").desc()) \
     .show(20, truncate=False)

edges.groupBy("receiver") \
     .agg(sum("pass_count").alias("received_passes")) \
     .orderBy(col("received_passes").desc()) \
     .show(20, truncate=False)

+-------------------------------+------------+
|passer                         |total_passes|
+-------------------------------+------------+
|Lionel Andrés Messi Cuccittini |31881       |
|Sergio Busquets i Burgos       |28070       |
|Xavier Hernández Creus         |22618       |
|Gerard Piqué Bernabéu          |20969       |
|Andrés Iniesta Luján           |19989       |
|Jordi Alba Ramos               |19387       |
|Daniel Alves da Silva          |17604       |
|Javier Alejandro Mascherano    |11962       |
|Ivan Rakitić                   |11906       |
|Sergi Roberto Carnicer         |9455        |
|Carles Puyol i Saforcada       |8850        |
|Neymar da Silva Santos Junior  |8021        |
|Francesc Fàbregas i Soler      |7989        |
|Samuel Yves Umtiti             |7231        |
|Granit Xhaka                   |6735        |
|Eric-Sylvain Bilal Abidal      |6605        |
|Pedro Eliezer Rodríguez Ledesma|6525        |
|Keira Walsh                    |6218        |
|Víctor Valdé

+-------------------------------+---------------+
|receiver                       |received_passes|
+-------------------------------+---------------+
|Lionel Andrés Messi Cuccittini |42144          |
|Sergio Busquets i Burgos       |24817          |
|Xavier Hernández Creus         |22205          |
|Andrés Iniesta Luján           |21536          |
|Gerard Piqué Bernabéu          |17647          |
|Jordi Alba Ramos               |16831          |
|Daniel Alves da Silva          |15718          |
|Ivan Rakitić                   |11512          |
|Neymar da Silva Santos Junior  |11084          |
|Javier Alejandro Mascherano    |10075          |
|Luis Alberto Suárez Díaz       |9328           |
|Sergi Roberto Carnicer         |8705           |
|Pedro Eliezer Rodríguez Ledesma|8654           |
|Francesc Fàbregas i Soler      |8383           |
|Carles Puyol i Saforcada       |7164           |
|Antoine Griezmann              |6331           |
|Granit Xhaka                   |6172           |


In [77]:
## en fazla pas yapan ikililer
edges.orderBy(col("pass_count").desc()).show(20, truncate=False)

+---------+------------------------------+-----------+------------------------------+-------+---------+----------+
|passer_id|passer                        |receiver_id|receiver                      |team_id|team     |pass_count|
+---------+------------------------------+-----------+------------------------------+-------+---------+----------+
|5203     |Sergio Busquets i Burgos      |5503       |Lionel Andrés Messi Cuccittini|217    |Barcelona|4155      |
|4324     |Daniel Alves da Silva         |5503       |Lionel Andrés Messi Cuccittini|217    |Barcelona|3967      |
|20131    |Xavier Hernández Creus        |5503       |Lionel Andrés Messi Cuccittini|217    |Barcelona|3314      |
|5503     |Lionel Andrés Messi Cuccittini|20131      |Xavier Hernández Creus        |217    |Barcelona|2634      |
|20131    |Xavier Hernández Creus        |4324       |Daniel Alves da Silva         |217    |Barcelona|2615      |
|4324     |Daniel Alves da Silva         |20131      |Xavier Hernández Creus    

In [78]:
events.inputFiles()[:5]

['file:///Users/onurercen/Desktop/401proje/statsbomb-open-data/data/events/3825839.json',
 'file:///Users/onurercen/Desktop/401proje/statsbomb-open-data/data/events/3901257.json',
 'file:///Users/onurercen/Desktop/401proje/statsbomb-open-data/data/events/3825690.json',
 'file:///Users/onurercen/Desktop/401proje/statsbomb-open-data/data/events/69267.json',
 'file:///Users/onurercen/Desktop/401proje/statsbomb-open-data/data/events/3795220.json']

## Events match_id ile yeniden oku


In [79]:
from pyspark.sql.functions import input_file_name, regexp_extract

events = (
    spark.read
    .option("multiline", True)
    .json("statsbomb-open-data/data/events/*.json")
    .withColumn("file_path", input_file_name())
    .withColumn(
        "match_id",
        regexp_extract("file_path", r"(\d+)\.json$", 1).cast("int")
    )
)

26/07/24 11:15:41 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: statsbomb-open-data/data/events/*.json.
java.io.FileNotFoundException: File statsbomb-open-data/data/events/*.json does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apac

In [80]:
events.select("match_id").show(10)

+--------+
|match_id|
+--------+
| 3942349|
| 3942349|
| 3942349|
| 3942349|
| 3942349|
| 3942349|
| 3942349|
| 3942349|
| 3942349|
| 3942349|
+--------+
only showing top 10 rows


In [81]:
events.select("match_id").distinct().count()

4235

In [82]:
matches.select("match_id").distinct().count()

4235

numbers match

In [83]:
## Reconstruct passes dataframe with match_id

from pyspark.sql.functions import col

passes = (
    events
    .filter(col("type.name") == "Pass")
    .select(
        "match_id",
        col("player.id").alias("passer_id"),
        col("player.name").alias("passer"),
        col("pass.recipient.id").alias("receiver_id"),
        col("pass.recipient.name").alias("receiver"),
        col("team.id").alias("team_id"),
        col("team.name").alias("team"),
        "minute",
        "second"
    )
    .filter(col("receiver").isNotNull())
    .cache()
)

passes.count()   # cache'i doldurur

26/07/24 11:16:24 WARN CacheManager: Asked to cache already cached data.


3847184

In [84]:
passes.printSchema()

root
 |-- match_id: integer (nullable = true)
 |-- passer_id: long (nullable = true)
 |-- passer: string (nullable = true)
 |-- receiver_id: long (nullable = true)
 |-- receiver: string (nullable = true)
 |-- team_id: long (nullable = true)
 |-- team: string (nullable = true)
 |-- minute: long (nullable = true)
 |-- second: long (nullable = true)



In [85]:
## Reconstruct edges dataframe with match_id
edges = (
    passes.groupBy(
        "match_id",
        "passer_id",
        "passer",
        "receiver_id",
        "receiver",
        "team_id",
        "team"
    )
    .count()
    .withColumnRenamed("count", "pass_count")
)

In [86]:
## Graph Validation

# Kaç oyuncu?
vertices.count()

# Kaç edge?
edges.count()

# Kaç maç?
edges.select("match_id").distinct().count()

4235

In [87]:
## En fazla pas yapan ikililer
edges.orderBy(
    col("pass_count").desc()
).show(20, truncate=False)

+--------+---------+--------------------------+-----------+------------------------------+-------+-------------------+----------+
|match_id|passer_id|passer                    |receiver_id|receiver                      |team_id|team               |pass_count|
+--------+---------+--------------------------+-----------+------------------------------+-------+-------------------+----------+
|3857255 |6765     |Rodrigo Hernández Cascante|6892       |Pau Francisco Torres          |772    |Spain              |69        |
|3869220 |6765     |Rodrigo Hernández Cascante|4353       |Aymeric Laporte               |772    |Spain              |68        |
|3857255 |6892     |Pau Francisco Torres      |6765       |Rodrigo Hernández Cascante    |772    |Spain              |67        |
|3775580 |4633     |Magdalena Lilly Eriksson  |4642       |Millie Bright                 |971    |Chelsea FCW        |63        |
|3893828 |4642     |Millie Bright             |10252      |Alex Greenwood                |

## Graph Analytics

In [88]:
from pyspark.sql.functions.builtin import countDistinct

out_degree = (
    edges
    .groupBy("passer_id", "passer")
    .agg(
        countDistinct("receiver_id").alias("out_degree")
    )
)

out_degree.orderBy("out_degree", ascending=False).show(20, truncate=False)

+---------+-------------------------------+----------+
|passer_id|passer                         |out_degree|
+---------+-------------------------------+----------+
|5503     |Lionel Andrés Messi Cuccittini |190       |
|5211     |Jordi Alba Ramos               |147       |
|5203     |Sergio Busquets i Burgos       |143       |
|6821     |Jesús Navas González           |126       |
|5470     |Ivan Rakitić                   |122       |
|5487     |Antoine Griezmann              |119       |
|5504     |Éver Maximiliano David Banega  |119       |
|5213     |Gerard Piqué Bernabéu          |115       |
|5201     |Sergio Ramos García            |114       |
|6867     |Papa Kouly Diop                |110       |
|6599     |Rubén Salvador Pérez Del Mármol|107       |
|6758     |Víctor Sánchez Mata            |103       |
|6720     |Pablo Sarabia García           |100       |
|6614     |Alexis Ruano Delgado           |100       |
|6913     |Fernando Navarro i Corbacho    |99        |
|26211    

In [89]:
##In-Degree Analysis
in_degree = (
    edges
    .groupBy("receiver_id", "receiver")
    .agg(
        countDistinct("passer_id").alias("in_degree")
    )
)

in_degree.orderBy("in_degree", ascending=False).show(20, truncate=False)

+-----------+-----------------------------------+---------+
|receiver_id|receiver                           |in_degree|
+-----------+-----------------------------------+---------+
|5503       |Lionel Andrés Messi Cuccittini     |204      |
|5203       |Sergio Busquets i Burgos           |141      |
|5211       |Jordi Alba Ramos                   |140      |
|6821       |Jesús Navas González               |135      |
|5487       |Antoine Griezmann                  |126      |
|5504       |Éver Maximiliano David Banega      |124      |
|5470       |Ivan Rakitić                       |118      |
|6720       |Pablo Sarabia García               |114      |
|6867       |Papa Kouly Diop                    |113      |
|6391       |Raúl García Escudero               |112      |
|5213       |Gerard Piqué Bernabéu              |110      |
|26211      |Joan Verdú Fernández               |109      |
|5201       |Sergio Ramos García                |105      |
|6651       |Joaquín Sánchez Rodríguez  

In [90]:
## TOTAL DEGREE
from pyspark.sql.functions import coalesce, col

degree = (
    out_degree.alias("o")
    .join(
        in_degree.alias("i"),
        col("o.passer_id") == col("i.receiver_id"),
        "full"
    )
    .select(
        coalesce(col("o.passer_id"), col("i.receiver_id")).alias("player_id"),
        coalesce(col("o.passer"), col("i.receiver")).alias("player"),
        coalesce(col("out_degree"), col("in_degree") * 0).alias("out_degree"),
        coalesce(col("in_degree"), col("out_degree") * 0).alias("in_degree")
    )
    .withColumn(
        "degree",
        col("out_degree") + col("in_degree")
    )
)

degree.orderBy(
    col("degree").desc()
).show(20, truncate=False)

+---------+-----------------------------------+----------+---------+------+
|player_id|player                             |out_degree|in_degree|degree|
+---------+-----------------------------------+----------+---------+------+
|5503     |Lionel Andrés Messi Cuccittini     |190       |204      |394   |
|5211     |Jordi Alba Ramos                   |147       |140      |287   |
|5203     |Sergio Busquets i Burgos           |143       |141      |284   |
|6821     |Jesús Navas González               |126       |135      |261   |
|5487     |Antoine Griezmann                  |119       |126      |245   |
|5504     |Éver Maximiliano David Banega      |119       |124      |243   |
|5470     |Ivan Rakitić                       |122       |118      |240   |
|5213     |Gerard Piqué Bernabéu              |115       |110      |225   |
|6867     |Papa Kouly Diop                    |110       |113      |223   |
|5201     |Sergio Ramos García                |114       |105      |219   |
|6720     |P

## NETWORK DENSITY

In [91]:
# NETWORK DENSITY

num_vertices = vertices.count()
num_edges = edges.count()

density = num_edges / (num_vertices * (num_vertices - 1))

print(f"Number of Players (Vertices): {num_vertices}")
print(f"Number of Passing Connections (Edges): {num_edges}")
print(f"Network Density: {density:.6f}")

Number of Players (Vertices): 13256
Number of Passing Connections (Edges): 997557
Network Density: 0.005677


## Network Density

Network density measures how many passing connections exist compared to the maximum possible number of connections in a directed network.

The density is calculated as:

Density = |E| / (|V| × (|V| − 1))

where:
- |E| is the number of edges (passing connections)
- |V| is the number of vertices (players)

A low density indicates that only a small fraction of all possible passing relationships actually occur, which is expected in football passing networks since players usually interact with only a subset of teammates.

In [92]:
## TOTAL PASSES BY TEAM
from pyspark.sql.functions import sum as spark_sum, col

team_passes = (
    edges
    .groupBy("team")
    .agg(
        spark_sum("pass_count").alias("total_passes")
    )
    .orderBy(col("total_passes").desc())
)

team_passes.show(truncate=False)

+--------------------------+------------+
|team                      |total_passes|
+--------------------------+------------+
|Barcelona                 |354639      |
|Paris Saint-Germain       |67704       |
|Manchester City WFC       |47231       |
|Arsenal WFC               |46505       |
|Chelsea FCW               |43624       |
|Arsenal                   |40020       |
|Bayer Leverkusen          |39467       |
|Manchester United         |38401       |
|Real Madrid               |35955       |
|Villarreal                |34414       |
|Everton LFC               |32886       |
|West Ham United LFC       |29557       |
|Brighton & Hove Albion WFC|28775       |
|Bayern Munich             |28721       |
|Atlético Madrid           |28026       |
|Napoli                    |27737       |
|Sevilla                   |26096       |
|Valencia                  |25776       |
|Athletic Club             |25173       |
|Celta Vigo                |24848       |
+--------------------------+------

In [93]:
## NUMBER OF PLAYERS PER TEAM
team_players = (
    vertices
    .groupBy("team")
    .count()
    .withColumnRenamed("count", "number_of_players")
    .orderBy(col("number_of_players").desc())
)

team_players.show(truncate=False)

+----------------------+-----------------+
|team                  |number_of_players|
+----------------------+-----------------+
|Barcelona             |158              |
|Sevilla               |148              |
|Real Betis            |148              |
|Getafe                |147              |
|Villarreal            |141              |
|Espanyol              |139              |
|Levante UD            |139              |
|Valencia              |132              |
|Málaga                |118              |
|Osasuna               |109              |
|Granada               |108              |
|Real Madrid           |105              |
|Atlético Madrid       |101              |
|Celta Vigo            |100              |
|RC Deportivo La Coruña|96               |
|Brazil                |96               |
|Real Valladolid       |95               |
|Manchester United     |92               |
|Athletic Club         |92               |
|Rayo Vallecano        |85               |
+----------

In [94]:
vertices.count()
vertices.select("id").distinct().count()

9914

In [95]:
vertices.filter(col("team") == "Barcelona").show(20, truncate=False)

+-----+------------------------------------------------+-------+---------+
|id   |name                                            |team_id|team     |
+-----+------------------------------------------------+-------+---------+
|5503 |Lionel Andrés Messi Cuccittini                  |217    |Barcelona|
|8118 |Frenkie de Jong                                 |217    |Barcelona|
|3478 |Francesc Fàbregas i Soler                       |217    |Barcelona|
|39073|Moriba Kourouma Kourouma                        |217    |Barcelona|
|5216 |Andrés Iniesta Luján                            |217    |Barcelona|
|24841|Ricard Puig Martí                               |217    |Barcelona|
|5477 |Ousmane Dembélé                                 |217    |Barcelona|
|30486|Pedro González López                            |217    |Barcelona|
|43728|Óscar Mingueza García                           |217    |Barcelona|
|5203 |Sergio Busquets i Burgos                        |217    |Barcelona|
|32480|Ronald Federico Ar

In [96]:
## AVG PASSES PER PLAYER
team_statistics = (
    team_passes.alias("p")
    .join(
        team_players.alias("t"),
        "team"
    )
    .withColumn(
        "avg_passes_per_player",
        col("total_passes") / col("number_of_players")
    )
    .orderBy(col("total_passes").desc())
)

team_statistics.show(truncate=False)

+--------------------------+------------+-----------------+---------------------+
|team                      |total_passes|number_of_players|avg_passes_per_player|
+--------------------------+------------+-----------------+---------------------+
|Barcelona                 |354639      |158              |2244.5506329113923   |
|Paris Saint-Germain       |67704       |64               |1057.875             |
|Manchester City WFC       |47231       |54               |874.6481481481482    |
|Arsenal WFC               |46505       |52               |894.3269230769231    |
|Chelsea FCW               |43624       |49               |890.2857142857143    |
|Arsenal                   |40020       |46               |870.0                |
|Bayer Leverkusen          |39467       |50               |789.34               |
|Manchester United         |38401       |92               |417.4021739130435    |
|Real Madrid               |35955       |105              |342.42857142857144   |
|Villarreal     

In [97]:
## top 10 team by passing activity
top10_teams = (
    team_passes
    .limit(10)
)

top10_teams.show(truncate=False)

+-------------------+------------+
|team               |total_passes|
+-------------------+------------+
|Barcelona          |354639      |
|Paris Saint-Germain|67704       |
|Manchester City WFC|47231       |
|Arsenal WFC        |46505       |
|Chelsea FCW        |43624       |
|Arsenal            |40020       |
|Bayer Leverkusen   |39467       |
|Manchester United  |38401       |
|Real Madrid        |35955       |
|Villarreal         |34414       |
+-------------------+------------+



In [98]:
##TOTAL SUCCESSFUL PASSES BY TEAM
from pyspark.sql.functions import sum as spark_sum, col

team_passes = (
    edges
    .groupBy("team")
    .agg(
        spark_sum("pass_count").alias("total_successful_passes")
    )
    .orderBy(col("total_successful_passes").desc())
)

team_passes.show(truncate=False)

+--------------------------+-----------------------+
|team                      |total_successful_passes|
+--------------------------+-----------------------+
|Barcelona                 |354639                 |
|Paris Saint-Germain       |67704                  |
|Manchester City WFC       |47231                  |
|Arsenal WFC               |46505                  |
|Chelsea FCW               |43624                  |
|Arsenal                   |40020                  |
|Bayer Leverkusen          |39467                  |
|Manchester United         |38401                  |
|Real Madrid               |35955                  |
|Villarreal                |34414                  |
|Everton LFC               |32886                  |
|West Ham United LFC       |29557                  |
|Brighton & Hove Albion WFC|28775                  |
|Bayern Munich             |28721                  |
|Atlético Madrid           |28026                  |
|Napoli                    |27737             

In [99]:
## avg passes per match by team
from pyspark.sql.functions import countDistinct

team_avg_passes = (
    edges
    .groupBy("team")
    .agg(
        spark_sum("pass_count").alias("total_passes"),
        countDistinct("match_id").alias("matches_played")
    )
    .withColumn(
        "average_passes_per_match",
        col("total_passes") / col("matches_played")
    )
    .orderBy(col("average_passes_per_match").desc())
)

team_avg_passes.show(truncate=False)

+-------------------+------------+--------------+------------------------+
|team               |total_passes|matches_played|average_passes_per_match|
+-------------------+------------+--------------+------------------------+
|Spain              |16545       |21            |787.8571428571429       |
|Barcelona WFC      |22876       |30            |762.5333333333333       |
|Bayern Munich      |28721       |40            |718.025                 |
|Paris Saint-Germain|67704       |95            |712.6736842105263       |
|Barcelona          |354639      |532           |666.6146616541354       |
|Napoli             |27737       |42            |660.4047619047619       |
|Spain Women's      |13692       |21            |652.0                   |
|Borussia Dortmund  |23555       |37            |636.6216216216217       |
|Germany            |10898       |18            |605.4444444444445       |
|Fiorentina         |23394       |39            |599.8461538461538       |
|Bayern München W   |1317

## Team Statistics

This section summarizes the passing activity of each team.

The following metrics are computed:

- Total successful passes by team
- Average successful passes per match
- Top 10 teams ranked by passing activity

These statistics provide an overview of team playing styles and overall passing performance across the dataset.

## Match Stats

In [100]:
## Read matches dataset

matches_df = (
    spark.read
    .option("multiline", "true")
    .json("statsbomb-open-data/data/matches/*/*.json")
)

print(f"Total number of matches: {matches_df.count()}")

matches_df.printSchema()

26/07/24 11:19:19 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: statsbomb-open-data/data/matches/*/*.json.
java.io.FileNotFoundException: File statsbomb-open-data/data/matches/*/*.json does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at or

Total number of matches: 4235
root
 |-- away_score: long (nullable = true)
 |-- away_team: struct (nullable = true)
 |    |-- away_team_gender: string (nullable = true)
 |    |-- away_team_group: string (nullable = true)
 |    |-- away_team_id: long (nullable = true)
 |    |-- away_team_name: string (nullable = true)
 |    |-- country: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |-- managers: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- country: struct (nullable = true)
 |    |    |    |    |-- id: long (nullable = true)
 |    |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- dob: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- nickname: string (nullable = true)
 |-- competition: struct (nullable = true)
 |    |-- competition_id: long (nullable = t

In [101]:
from pyspark.sql import functions as F

matches = matches_df.select(
    F.col("match_id"),
    F.col("match_date"),
    F.col("competition.competition_name").alias("competition"),
    F.col("season.season_name").alias("season"),
    F.col("home_team.home_team_name").alias("home_team"),
    F.col("away_team.away_team_name").alias("away_team"),
    F.col("home_score"),
    F.col("away_score")
)

matches.show(5, truncate=False)

+--------+----------+-----------+---------+----------------------+----------+----------+----------+
|match_id|match_date|competition|season   |home_team             |away_team |home_score|away_score|
+--------+----------+-----------+---------+----------------------+----------+----------+----------+
|3825848 |2015-09-23|La Liga    |2015/2016|Levante UD            |Eibar     |2         |2         |
|3825895 |2015-09-23|La Liga    |2015/2016|Las Palmas            |Sevilla   |2         |0         |
|3825894 |2016-05-01|La Liga    |2015/2016|RC Deportivo La Coruña|Getafe    |0         |2         |
|3825855 |2016-05-02|La Liga    |2015/2016|Málaga                |Levante UD|3         |1         |
|3825908 |2016-05-15|La Liga    |2015/2016|Espanyol              |Eibar     |4         |2         |
+--------+----------+-----------+---------+----------------------+----------+----------+----------+
only showing top 5 rows


In [102]:
match_passes = (
    edges.join(matches, on="match_id", how="left")
)

match_passes.show(5, truncate=False)

+--------+---------+-------------------------------+-----------+---------------------------------------+-------+------+----------+----------+--------------+------+-----------+---------+----------+----------+
|match_id|passer_id|passer                         |receiver_id|receiver                               |team_id|team  |pass_count|match_date|competition   |season|home_team  |away_team|home_score|away_score|
+--------+---------+-------------------------------+-----------+---------------------------------------+-------+------+----------+----------+--------------+------+-----------+---------+----------+----------+
|3942349 |6704     |Theo Bernard François Hernández|11990      |Youssouf Fofana                        |771    |France|1         |2024-07-05|UEFA Euro     |2024  |Portugal   |France   |0         |0         |
|3869420 |22600    |Lucas Tolentino Coelho de Lima |18395      |Vinícius José Paixão de Oliveira Júnior|781    |Brazil|2         |2022-12-09|FIFA World Cup|2022  |Croat

In [103]:
## En fazla pas yapılan maçlar

from pyspark.sql import functions as F

match_total_passes = (
    match_passes
    .groupBy(
        "match_id",
        "match_date",
        "competition",
        "season",
        "home_team",
        "away_team"
    )
    .agg(
        F.sum("pass_count").alias("total_passes")
    )
    .orderBy(F.desc("total_passes"))
)

match_total_passes.show(10, truncate=False)

+--------+----------+--------------+---------+-------------------+-------------------+------------+
|match_id|match_date|competition   |season   |home_team          |away_team          |total_passes|
+--------+----------+--------------+---------+-------------------+-------------------+------------+
|3942349 |2024-07-05|UEFA Euro     |2024     |Portugal           |France             |1518        |
|3794692 |2021-06-29|UEFA Euro     |2020     |Sweden             |Ukraine            |1445        |
|3869420 |2022-12-09|FIFA World Cup|2022     |Croatia            |Brazil             |1418        |
|7582    |2018-07-01|FIFA World Cup|2018     |Spain              |Russia             |1409        |
|3795108 |2021-07-02|UEFA Euro     |2020     |Switzerland        |Spain              |1386        |
|3794686 |2021-06-28|UEFA Euro     |2020     |Croatia            |Spain              |1355        |
|3837938 |2023-04-08|Ligue 1       |2022/2023|OGC Nice           |Paris Saint-Germain|1340        |


In [104]:
match_total_passes.describe().show()

+-------+------------------+----------+-----------------+------------------+--------------+--------------+-----------------+
|summary|          match_id|match_date|      competition|            season|     home_team|     away_team|     total_passes|
+-------+------------------+----------+-----------------+------------------+--------------+--------------+-----------------+
|  count|              4235|      4235|             4235|              4235|          4235|          4235|             4235|
|   mean| 3154084.057615112|      NULL|             NULL|2020.2727272727273|          NULL|          NULL| 908.425974025974|
| stddev|1446547.7857588269|      NULL|             NULL| 8.782985270851732|          NULL|          NULL|134.0313924328946|
|    min|              7298|1958-06-24|    1. Bundesliga|              1958|    AC Ajaccio|    AC Ajaccio|              461|
|    max|           4020846|2025-07-27|Women's World Cup|              2025|Zambia Women's|Zambia Women's|             1518|


In [105]:
## Takımların maç başına pas sayısı
team_match_passes = (
    match_passes
    .groupBy(
        "match_id",
        "match_date",
        "competition",
        "home_team",
        "away_team",
        "team"
    )
    .agg(
        F.sum("pass_count").alias("passes")
    )
    .orderBy(F.desc("passes"))
)

team_match_passes.show(20, truncate=False)

+--------+----------+--------------+-------------------+------------------------+-------------------+------+
|match_id|match_date|competition   |home_team          |away_team               |team               |passes|
+--------+----------+--------------+-------------------+------------------------+-------------------+------+
|7582    |2018-07-01|FIFA World Cup|Spain              |Russia                  |Spain              |1113  |
|3857255 |2022-12-01|FIFA World Cup|Japan              |Spain                   |Spain              |1081  |
|70280   |2012-11-25|La Liga       |Levante UD         |Barcelona               |Barcelona          |1077  |
|69272   |2011-05-11|La Liga       |Levante UD         |Barcelona               |Barcelona          |1069  |
|3857291 |2022-11-23|FIFA World Cup|Spain              |Costa Rica              |Spain              |1062  |
|303610  |2020-01-19|La Liga       |Barcelona          |Granada                 |Barcelona          |1019  |
|69293   |2011-10-1

Observation: Teams with higher total pass counts generally tend to adopt a possession-oriented style of play, emphasizing ball retention, build-up play, and controlled attacks. In contrast, teams with lower pass counts may rely on a more direct or defensive approach, often focusing on compact defending and quick counterattacks rather than prolonged possession. However, passing volume alone is not sufficient to classify a team's tactical approach, as match context, opponent quality, and game state can significantly influence passing behavior.

In [106]:
## KAÇ FARKLI PASSER -> RECEIVER İKİLİSİ VAR?
unique_connections = (
    match_passes
    .groupBy(
        "match_id",
        "match_date",
        "home_team",
        "away_team"
    )
    .agg(
        F.count("*").alias("unique_passing_connections")
    )
    .orderBy(F.desc("unique_passing_connections"))
)

unique_connections.show(20, truncate=False)

+--------+----------+------------------------+-------------------+--------------------------+
|match_id|match_date|home_team               |away_team          |unique_passing_connections|
+--------+----------+------------------------+-------------------+--------------------------+
|3794686 |2021-06-28|Croatia                 |Spain              |332                       |
|3795221 |2021-07-07|England                 |Denmark            |317                       |
|3942226 |2024-07-05|Spain                   |Germany            |316                       |
|3795220 |2021-07-06|Italy                   |Spain              |315                       |
|3869321 |2022-12-09|Netherlands             |Argentina          |311                       |
|3922659 |2024-02-03|Mali                    |Côte d'Ivoire      |306                       |
|3795108 |2021-07-02|Switzerland             |Spain              |306                       |
|3941017 |2024-06-30|England                 |Slovakia      

In [107]:
## Her maçta pas yapan oyuncu sayısı
players_per_match = (
    match_passes
    .groupBy(
        "match_id",
        "match_date",
        "home_team",
        "away_team"
    )
    .agg(
        F.countDistinct("passer_id").alias("players_involved")
    )
    .orderBy(F.desc("players_involved"))
)

players_per_match.show(20, truncate=False)

+--------+----------+------------------+-------------------+----------------+
|match_id|match_date|home_team         |away_team          |players_involved|
+--------+----------+------------------+-------------------+----------------+
|3794692 |2021-06-29|Sweden            |Ukraine            |34              |
|3795220 |2021-07-06|Italy             |Spain              |34              |
|3922658 |2024-02-03|Cape Verde Islands|South Africa       |34              |
|3794686 |2021-06-28|Croatia           |Spain              |34              |
|3795108 |2021-07-02|Switzerland       |Spain              |34              |
|3869685 |2022-12-18|Argentina         |France             |34              |
|3847567 |2022-07-31|England Women's   |Germany Women's    |33              |
|3869219 |2022-12-05|Japan             |Croatia            |33              |
|3943077 |2024-07-15|Argentina         |Colombia           |33              |
|3922659 |2024-02-03|Mali              |Côte d'Ivoire      |33  

In [108]:
## Maç başına istatistikler - min max. avg vs
match_summary = (
    match_total_passes
    .select("total_passes")
    .summary(
        "count",
        "mean",
        "stddev",
        "min",
        "max"
    )
)

match_summary.show()

+-------+-----------------+
|summary|     total_passes|
+-------+-----------------+
|  count|             4235|
|   mean| 908.425974025974|
| stddev|134.0313924328946|
|    min|              461|
|    max|             1518|
+-------+-----------------+



## PLAYER STATS

In [109]:
## En çok pas atan oyuncular
from pyspark.sql import functions as F

top_passers = (
    edges
    .groupBy("passer_id", "passer")
    .agg(
        F.sum("pass_count").alias("total_passes")
    )
    .orderBy(F.desc("total_passes"))
)

top_passers.show(20, truncate=False)

+---------+-------------------------------+------------+
|passer_id|passer                         |total_passes|
+---------+-------------------------------+------------+
|5503     |Lionel Andrés Messi Cuccittini |31881       |
|5203     |Sergio Busquets i Burgos       |28070       |
|20131    |Xavier Hernández Creus         |22618       |
|5213     |Gerard Piqué Bernabéu          |20969       |
|5216     |Andrés Iniesta Luján           |19989       |
|5211     |Jordi Alba Ramos               |19387       |
|4324     |Daniel Alves da Silva          |17604       |
|5506     |Javier Alejandro Mascherano    |11962       |
|5470     |Ivan Rakitić                   |11906       |
|6379     |Sergi Roberto Carnicer         |9455        |
|20125    |Carles Puyol i Saforcada       |8850        |
|4320     |Neymar da Silva Santos Junior  |8021        |
|3478     |Francesc Fàbregas i Soler      |7989        |
|5492     |Samuel Yves Umtiti             |7231        |
|3500     |Granit Xhaka        

In [110]:
## En çok pas alan oyuncular
top_receivers = (
    edges
    .groupBy("receiver_id", "receiver")
    .agg(
        F.sum("pass_count").alias("received_passes")
    )
    .orderBy(F.desc("received_passes"))
)
top_receivers.show(20, truncate=False)

+-----------+-------------------------------+---------------+
|receiver_id|receiver                       |received_passes|
+-----------+-------------------------------+---------------+
|5503       |Lionel Andrés Messi Cuccittini |42144          |
|5203       |Sergio Busquets i Burgos       |24817          |
|20131      |Xavier Hernández Creus         |22205          |
|5216       |Andrés Iniesta Luján           |21536          |
|5213       |Gerard Piqué Bernabéu          |17647          |
|5211       |Jordi Alba Ramos               |16831          |
|4324       |Daniel Alves da Silva          |15718          |
|5470       |Ivan Rakitić                   |11512          |
|4320       |Neymar da Silva Santos Junior  |11084          |
|5506       |Javier Alejandro Mascherano    |10075          |
|5246       |Luis Alberto Suárez Díaz       |9328           |
|6379       |Sergi Roberto Carnicer         |8705           |
|3958       |Pedro Eliezer Rodríguez Ledesma|8654           |
|3478   

In [115]:
## Oyuncu katılımı (pas alma + pas verme)
sent = (
    edges
    .groupBy("passer_id", "passer")
    .agg(F.sum("pass_count").alias("passes_sent"))
)

received = (
    edges
    .groupBy("receiver_id", "receiver")
    .agg(F.sum("pass_count").alias("passes_received"))
    .withColumnRenamed("receiver_id", "player_id")
    .withColumnRenamed("receiver", "player")
)

sent = (
    sent
    .withColumnRenamed("passer_id", "player_id")
    .withColumnRenamed("passer", "player")
)

player_involvement = (
    sent.join(received, ["player_id", "player"], "outer")
    .fillna(0)
    .withColumn(
        "total_involvement",
        F.col("passes_sent") + F.col("passes_received")
    )
    .orderBy(F.desc("total_involvement"))
)

player_involvement.show(20, truncate=False)

+---------+-------------------------------+-----------+---------------+-----------------+
|player_id|player                         |passes_sent|passes_received|total_involvement|
+---------+-------------------------------+-----------+---------------+-----------------+
|5503     |Lionel Andrés Messi Cuccittini |31881      |42144          |74025            |
|5203     |Sergio Busquets i Burgos       |28070      |24817          |52887            |
|20131    |Xavier Hernández Creus         |22618      |22205          |44823            |
|5216     |Andrés Iniesta Luján           |19989      |21536          |41525            |
|5213     |Gerard Piqué Bernabéu          |20969      |17647          |38616            |
|5211     |Jordi Alba Ramos               |19387      |16831          |36218            |
|4324     |Daniel Alves da Silva          |17604      |15718          |33322            |
|5470     |Ivan Rakitić                   |11906      |11512          |23418            |
|5506     

In [116]:
## En güçlü oyuncu ikilileri
top_pairs = (
    edges
    .groupBy(
        "passer",
        "receiver"
    )
    .agg(
        F.sum("pass_count").alias("passes_between")
    )
    .orderBy(F.desc("passes_between"))
)

top_pairs.show(20, truncate=False)

+------------------------------+------------------------------+--------------+
|passer                        |receiver                      |passes_between|
+------------------------------+------------------------------+--------------+
|Sergio Busquets i Burgos      |Lionel Andrés Messi Cuccittini|4238          |
|Daniel Alves da Silva         |Lionel Andrés Messi Cuccittini|3967          |
|Xavier Hernández Creus        |Lionel Andrés Messi Cuccittini|3314          |
|Lionel Andrés Messi Cuccittini|Xavier Hernández Creus        |2634          |
|Xavier Hernández Creus        |Daniel Alves da Silva         |2615          |
|Daniel Alves da Silva         |Xavier Hernández Creus        |2579          |
|Andrés Iniesta Luján          |Lionel Andrés Messi Cuccittini|2562          |
|Lionel Andrés Messi Cuccittini|Andrés Iniesta Luján          |2358          |
|Lionel Andrés Messi Cuccittini|Daniel Alves da Silva         |2288          |
|Lionel Andrés Messi Cuccittini|Sergio Busquets i Bu

In [117]:
## En fazla farklı oyuncu ile bağlantı kuranlar
most_connections = (
    edges
    .groupBy(
        "passer_id",
        "passer"
    )
    .agg(
        F.countDistinct("receiver_id").alias("unique_teammates")
    )
    .orderBy(F.desc("unique_teammates"))
)

most_connections.show(20, truncate=False)

+---------+-------------------------------+----------------+
|passer_id|passer                         |unique_teammates|
+---------+-------------------------------+----------------+
|5503     |Lionel Andrés Messi Cuccittini |190             |
|5211     |Jordi Alba Ramos               |147             |
|5203     |Sergio Busquets i Burgos       |143             |
|6821     |Jesús Navas González           |126             |
|5470     |Ivan Rakitić                   |122             |
|5487     |Antoine Griezmann              |119             |
|5504     |Éver Maximiliano David Banega  |119             |
|5213     |Gerard Piqué Bernabéu          |115             |
|5201     |Sergio Ramos García            |114             |
|6867     |Papa Kouly Diop                |110             |
|6599     |Rubén Salvador Pérez Del Mármol|107             |
|6758     |Víctor Sánchez Mata            |103             |
|6720     |Pablo Sarabia García           |100             |
|6614     |Alexis Ruano 

In [118]:
## En fazla farklı oyuncudan pas alanlar
most_received_connections = (
    edges
    .groupBy(
        "receiver_id",
        "receiver"
    )
    .agg(
        F.countDistinct("passer_id").alias("unique_passers")
    )
    .orderBy(F.desc("unique_passers"))
)

most_received_connections.show(20, truncate=False)

+-----------+-----------------------------------+--------------+
|receiver_id|receiver                           |unique_passers|
+-----------+-----------------------------------+--------------+
|5503       |Lionel Andrés Messi Cuccittini     |204           |
|5203       |Sergio Busquets i Burgos           |141           |
|5211       |Jordi Alba Ramos                   |140           |
|6821       |Jesús Navas González               |135           |
|5487       |Antoine Griezmann                  |126           |
|5504       |Éver Maximiliano David Banega      |124           |
|5470       |Ivan Rakitić                       |118           |
|6720       |Pablo Sarabia García               |114           |
|6867       |Papa Kouly Diop                    |113           |
|6391       |Raúl García Escudero               |112           |
|5213       |Gerard Piqué Bernabéu              |110           |
|26211      |Joan Verdú Fernández               |109           |
|5201       |Sergio Ramos

In [119]:
top_passers.select("total_passes").summary(
    "count", ##oyıuncu sayısı
    "mean", ## oyuncu başına ortalama pas sayısı
    "stddev", ## standart sapma
    "min", ## en az pas atan oyuncu
    "max" ## en çok pas atan oyuncu
).show()

+-------+------------------+
|summary|      total_passes|
+-------+------------------+
|  count|              9931|
|   mean|387.39140066458566|
| stddev|  882.328031088021|
|    min|                 1|
|    max|             31881|
+-------+------------------+

